# 正則化與數據增強技術

深度學習模型容易出現**過擬合**問題，即在訓練集上表現很好，但在測試集上表現較差。本章介紹多種防止過擬合的技術。

## 本章內容

### 正則化技術
1. **Dropout** - 隨機失活
2. **Batch Normalization** - 批量歸一化
3. **Layer Normalization** - 層歸一化
4. **Weight Decay (L2正則化)** - 權重衰減
5. **Early Stopping** - 早停法

### 數據增強技術
1. **基礎變換** - 翻轉、旋轉、裁剪
2. **顏色變換** - 亮度、對比度、飽和度調整
3. **進階技術** - Cutout, Mixup, CutMix
4. **自動增強** - AutoAugment, RandAugment

## 學習目標

- 理解過擬合的原因和表現
- 掌握各種正則化技術的原理和實現
- 學會使用數據增強提升模型泛化能力
- 了解如何選擇和組合不同的技術

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import copy

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 設備配置
device = torch.device('cuda' if torch.cuda.is_available() else 
                     'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'使用設備: {device}')

# 設置matplotlib中文顯示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 第一部分：正則化技術

## 1. Dropout (隨機失活)

### 原理

Dropout在訓練時隨機「關閉」一部分神經元，迫使網路不依賴特定的神經元，從而提高泛化能力。

```
訓練時：
輸入 → [N1, N2, N3, N4, N5] → 隨機失活 → [N1, ✗, N3, ✗, N5] → 輸出

測試時：
輸入 → [N1, N2, N3, N4, N5] → 全部保留 → 輸出 (按比例縮放)
```

### 工作機制

- **訓練階段**：以概率 p 隨機將神經元輸出置為0
- **測試階段**：使用所有神經元，但輸出乘以 (1-p)
- **效果**：相當於訓練多個子網路的集成模型

### 優勢

1. 防止神經元共適應(co-adaptation)
2. 減少過擬合
3. 提供模型正則化
4. 近似模型集成

### 使用建議

- 全連接層通常使用 p=0.5
- 卷積層使用較小的 p (0.1-0.3)
- 輸入層使用更小的 p (0.1-0.2)

In [ ]:
class DropoutDemo(nn.Module):
    """演示Dropout的效果"""
    
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, dropout_rate=0.5):
        super(DropoutDemo, self).__init__()
        
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.dropout1 = nn.Dropout(p=dropout_rate)
        
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.dropout2 = nn.Dropout(p=dropout_rate)
        
        self.fc3 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # 展平
        
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)  # 第一個dropout
        
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)  # 第二個dropout
        
        x = self.fc3(x)
        return x

# 測試Dropout在訓練和評估模式下的不同行為
model = DropoutDemo(dropout_rate=0.5)
x = torch.randn(1, 1, 28, 28)

print("=== Dropout行為測試 ===")
print("\n訓練模式（Dropout啟用）:")
model.train()
for i in range(3):
    output = model(x)
    print(f"第{i+1}次前向傳播: {output[0][:5]}...")  # 每次結果不同

print("\n評估模式（Dropout關閉）:")
model.eval()
with torch.no_grad():
    for i in range(3):
        output = model(x)
        print(f"第{i+1}次前向傳播: {output[0][:5]}...")  # 每次結果相同

In [ ]:
# 手動實現Dropout
def dropout_layer(X, dropout_rate):
    """
    手動實現Dropout層
    
    Args:
        X: 輸入張量
        dropout_rate: dropout比率
    """
    assert 0 <= dropout_rate <= 1
    
    # dropout_rate = 1 時，所有元素都被丟棄
    if dropout_rate == 1:
        return torch.zeros_like(X)
    
    # dropout_rate = 0 時，所有元素都被保留
    if dropout_rate == 0:
        return X
    
    # 生成隨機mask (0或1)
    mask = (torch.rand(X.shape) > dropout_rate).float()
    
    # 應用mask並縮放
    return mask * X / (1.0 - dropout_rate)

# 測試手動實現的Dropout
X = torch.arange(16, dtype=torch.float32).reshape((2, 8))
print("原始輸入:")
print(X)

print("\nDropout rate = 0 (保留所有):")
print(dropout_layer(X, 0))

print("\nDropout rate = 0.5:")
print(dropout_layer(X, 0.5))

print("\nDropout rate = 1 (丟棄所有):")
print(dropout_layer(X, 1))

## 2. Batch Normalization (批量歸一化)

### 問題：內部協變量偏移

在深度網路中，每層的輸入分佈會隨著前面層參數的更新而改變，這種現象稱為**內部協變量偏移**(Internal Covariate Shift)。

### Batch Normalization的解決方案

對每個小批量的數據進行歸一化：

$$\mu_B = \frac{1}{m}\sum_{i=1}^m x_i$$

$$\sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_B)^2$$

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

$$y_i = \gamma \hat{x}_i + \beta$$

其中 γ 和 β 是可學習的參數。

### 優勢

1. **加速訓練**：可以使用更大的學習率
2. **減少對初始化的依賴**
3. **正則化效果**：減少對Dropout的需求
4. **允許更深的網路**：緩解梯度消失

### 使用位置

通常在激活函數**之前**使用：
```
Conv/Linear → BatchNorm → ReLU
```

In [ ]:
# 手動實現Batch Normalization
def batch_norm(X, gamma, beta, moving_mean, moving_var, eps=1e-5, momentum=0.9):
    """
    手動實現批量歸一化
    
    Args:
        X: 輸入數據
        gamma: 縮放參數
        beta: 偏移參數
        moving_mean: 移動平均值（用於推理）
        moving_var: 移動方差（用於推理）
        eps: 防止除零的小常數
        momentum: 移動平均的動量
    """
    # 判斷是訓練模式還是預測模式
    if not torch.is_grad_enabled():
        # 預測模式：使用移動平均的均值和方差
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
    else:
        # 訓練模式：計算當前批次的均值和方差
        assert len(X.shape) in (2, 4)  # 全連接或卷積層
        
        if len(X.shape) == 2:
            # 全連接層：對特徵維度計算
            mean = X.mean(dim=0)
            var = ((X - mean) ** 2).mean(dim=0)
        else:
            # 卷積層：對批量、高度、寬度計算，保留通道維度
            mean = X.mean(dim=(0, 2, 3), keepdim=True)
            var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)
        
        # 歸一化
        X_hat = (X - mean) / torch.sqrt(var + eps)
        
        # 更新移動平均
        moving_mean = momentum * moving_mean + (1.0 - momentum) * mean
        moving_var = momentum * moving_var + (1.0 - momentum) * var
    
    # 縮放和偏移
    Y = gamma * X_hat + beta
    return Y, moving_mean.data, moving_var.data

# 測試Batch Normalization
X = torch.randn(4, 3, 8, 8)  # (batch_size, channels, height, width)
print("輸入統計:")
print(f"均值: {X.mean():.4f}")
print(f"標準差: {X.std():.4f}")

# 初始化參數
gamma = torch.ones(3)
beta = torch.zeros(3)
moving_mean = torch.zeros(3)
moving_var = torch.ones(3)

# 進行歸一化
gamma = gamma.reshape(1, 3, 1, 1)
beta = beta.reshape(1, 3, 1, 1)
moving_mean = moving_mean.reshape(1, 3, 1, 1)
moving_var = moving_var.reshape(1, 3, 1, 1)

Y, moving_mean, moving_var = batch_norm(X, gamma, beta, moving_mean, moving_var)

print("\nBatch Norm 後的統計:")
print(f"均值: {Y.mean():.4f}")
print(f"標準差: {Y.std():.4f}")

In [ ]:
# 使用PyTorch的Batch Normalization
class ConvNetWithBN(nn.Module):
    """帶有Batch Normalization的卷積網路"""
    
    def __init__(self, num_classes=10):
        super(ConvNetWithBN, self).__init__()
        
        # 第一個卷積塊
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)  # Batch Norm for 2D
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # 第二個卷積塊
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # 全連接層
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.bn3 = nn.BatchNorm1d(128)  # Batch Norm for 1D
        self.relu3 = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # Conv Block 1: Conv → BN → ReLU → Pool
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # Conv Block 2: Conv → BN → ReLU → Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC: Linear → BN → ReLU → Dropout
        x = self.fc1(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        return x

# 創建模型
model_with_bn = ConvNetWithBN()
print("=== 帶有Batch Normalization的模型 ===")
print(model_with_bn)

# 測試
x = torch.randn(2, 1, 28, 28)
output = model_with_bn(x)
print(f"\n輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")

## 3. 其他正則化技術

### Layer Normalization

與Batch Normalization類似，但是對每個樣本的所有特徵進行歸一化，而不是對批次進行歸一化。

**優勢**：
- 不依賴批次大小
- 適用於RNN和小批量情況
- 在Transformer中廣泛使用

### Weight Decay (L2正則化)

在損失函數中添加權重的L2範數：

$$L_{total} = L_{original} + \frac{\lambda}{2}\sum_i w_i^2$$

**效果**：限制權重大小，防止過擬合

### Early Stopping

監控驗證集性能，當性能不再提升時停止訓練。

**實施步驟**：
1. 監控驗證損失
2. 保存最佳模型
3. 設置耐心值(patience)
4. 超過耐心值時停止訓練

In [ ]:
# Layer Normalization 示例
class ModelWithLayerNorm(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, num_classes=10):
        super(ModelWithLayerNorm, self).__init__()
        
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.ln1 = nn.LayerNorm(hidden_size)  # Layer Normalization
        
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        
        self.fc3 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        
        x = self.fc1(x)
        x = self.ln1(x)  # Layer Norm
        x = F.relu(x)
        
        x = self.fc2(x)
        x = self.ln2(x)  # Layer Norm
        x = F.relu(x)
        
        x = self.fc3(x)
        return x

# Early Stopping 實現
class EarlyStopping:
    """早停法實現"""
    
    def __init__(self, patience=7, verbose=True, delta=0):
        """
        Args:
            patience: 容忍多少個epoch沒有改善
            verbose: 是否打印信息
            delta: 最小改善量
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.best_model = None
    
    def __call__(self, val_loss, model):
        score = -val_loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
    
    def save_checkpoint(self, val_loss, model):
        """保存模型"""
        if self.verbose:
            print(f'驗證損失降低 ({self.val_loss_min:.6f} --> {val_loss:.6f}). 保存模型...')
        self.best_model = copy.deepcopy(model.state_dict())
        self.val_loss_min = val_loss

print("Early Stopping類已定義，可在訓練循環中使用。")

## 第二部分：數據增強技術

## 4. 基礎數據增強

數據增強通過對訓練數據進行變換來人工擴充數據集，提高模型的泛化能力。

### 常見的基礎變換

1. **幾何變換**
   - 隨機裁剪 (Random Crop)
   - 隨機翻轉 (Random Flip)
   - 隨機旋轉 (Random Rotation)
   - 隨機縮放 (Random Scale)

2. **顏色變換**
   - 亮度調整 (Brightness)
   - 對比度調整 (Contrast)
   - 飽和度調整 (Saturation)
   - 色相調整 (Hue)

3. **噪聲和模糊**
   - 高斯噪聲
   - 高斯模糊
   - 運動模糊

In [ ]:
# 載入示例數據
from torchvision.datasets import FashionMNIST

# 基礎數據增強示例
basic_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  # 50%概率水平翻轉
    transforms.RandomRotation(10),  # 隨機旋轉±10度
    transforms.RandomCrop(28, padding=4),  # 先padding再隨機裁剪
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # 歸一化
])

# 載入一張圖像進行演示
dataset = FashionMNIST(root='./data', train=True, download=True, transform=None)
original_img, label = dataset[0]

# 應用多次變換查看效果
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.ravel()

# 顯示原圖
axes[0].imshow(original_img, cmap='gray')
axes[0].set_title('原始圖像')
axes[0].axis('off')

# 應用不同的增強
augmentations = [
    ('水平翻轉', transforms.RandomHorizontalFlip(p=1.0)),
    ('旋轉15度', transforms.RandomRotation(15)),
    ('隨機裁剪', transforms.RandomCrop(28, padding=4)),
    ('亮度調整', transforms.ColorJitter(brightness=0.5)),
    ('對比度調整', transforms.ColorJitter(contrast=0.5)),
    ('組合變換', transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.RandomCrop(28, padding=4)
    ]))
]

for idx, (name, transform) in enumerate(augmentations, 1):
    augmented = transform(original_img)
    axes[idx].imshow(augmented, cmap='gray')
    axes[idx].set_title(name)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('basic_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

print("基礎數據增強示例已生成！")

## 5. 進階數據增強技術

### Cutout

隨機遮蔽圖像的一部分，迫使模型學習更魯棒的特徵。

```
原圖: [完整的貓]  →  Cutout: [部分被遮蔽的貓]
```

### Mixup

混合兩張圖像及其標籤：

$$x' = \lambda x_i + (1-\lambda) x_j$$
$$y' = \lambda y_i + (1-\lambda) y_j$$

其中 λ ~ Beta(α, α)

### CutMix

結合Cutout和Mixup：將一張圖像的一部分替換為另一張圖像。

### Random Erasing

與Cutout類似，但使用隨機值填充而不是固定值。

In [ ]:
class Cutout:
    """Cutout數據增強"""
    
    def __init__(self, n_holes=1, length=16):
        """
        Args:
            n_holes: 遮蔽區域的數量
            length: 遮蔽區域的邊長
        """
        self.n_holes = n_holes
        self.length = length
    
    def __call__(self, img):
        """
        Args:
            img: Tensor圖像，形狀為(C, H, W)
        """
        h, w = img.size(1), img.size(2)
        mask = np.ones((h, w), np.float32)
        
        for n in range(self.n_holes):
            # 隨機選擇遮蔽中心
            y = np.random.randint(h)
            x = np.random.randint(w)
            
            # 計算遮蔽區域的邊界
            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)
            
            # 遮蔽該區域
            mask[y1:y2, x1:x2] = 0.
        
        mask = torch.from_numpy(mask)
        mask = mask.expand_as(img)
        img = img * mask
        
        return img

def mixup_data(x, y, alpha=1.0):
    """執行Mixup數據增強"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup的損失函數"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# 測試Cutout
cutout = Cutout(n_holes=1, length=8)

# 創建示例圖像
img_tensor = transforms.ToTensor()(original_img)

# 應用Cutout
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(original_img, cmap='gray')
axes[0].set_title('原始圖像')
axes[0].axis('off')

for i in range(4):
    cutout_img = cutout(img_tensor.clone())
    axes[i+1].imshow(cutout_img.squeeze(), cmap='gray')
    axes[i+1].set_title(f'Cutout {i+1}')
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig('cutout_example.png', dpi=150, bbox_inches='tight')
plt.show()

print("Cutout增強示例已生成！")

In [ ]:
# Mixup視覺化示例
# 載入兩張不同的圖像
img1, label1 = dataset[0]
img2, label2 = dataset[1]

# 轉換為張量
img1_tensor = transforms.ToTensor()(img1).unsqueeze(0)
img2_tensor = transforms.ToTensor()(img2).unsqueeze(0)

# 應用不同的lambda值
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

lambdas = [0, 0.25, 0.5, 0.75]

# 第一行：原圖和不同lambda的mixup
for i, lam in enumerate(lambdas):
    mixed = lam * img1_tensor + (1 - lam) * img2_tensor
    axes[0, i].imshow(mixed.squeeze(), cmap='gray')
    axes[0, i].set_title(f'λ = {lam}')
    axes[0, i].axis('off')

# 第二行：更多隨機樣本
for i in range(4):
    lam = np.random.beta(1.0, 1.0)
    mixed = lam * img1_tensor + (1 - lam) * img2_tensor
    axes[1, i].imshow(mixed.squeeze(), cmap='gray')
    axes[1, i].set_title(f'λ = {lam:.2f}')
    axes[1, i].axis('off')

plt.suptitle('Mixup數據增強示例', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('mixup_example.png', dpi=150, bbox_inches='tight')
plt.show()

print("Mixup增強示例已生成！")

## 6. 完整的訓練示例：結合正則化和數據增強

下面展示如何在實際訓練中結合使用這些技術：

In [ ]:
# 定義完整的訓練和驗證函數
def train_with_regularization(model, train_loader, val_loader, 
                             num_epochs=10, learning_rate=0.001,
                             weight_decay=1e-4, use_mixup=False):
    """
    使用正則化技術訓練模型
    
    Args:
        model: 要訓練的模型
        train_loader: 訓練數據加載器
        val_loader: 驗證數據加載器
        num_epochs: 訓練輪數
        learning_rate: 學習率
        weight_decay: 權重衰減係數
        use_mixup: 是否使用Mixup
    """
    # 定義優化器（包含weight decay）
    optimizer = torch.optim.Adam(model.parameters(), 
                                lr=learning_rate,
                                weight_decay=weight_decay)
    
    # 學習率調度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    # 損失函數
    criterion = nn.CrossEntropyLoss()
    
    # Early Stopping
    early_stopping = EarlyStopping(patience=5, verbose=True)
    
    # 記錄歷史
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    model = model.to(device)
    
    for epoch in range(num_epochs):
        # 訓練階段
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            if use_mixup:
                # 使用Mixup
                data, target_a, target_b, lam = mixup_data(data, target, alpha=1.0)
                
                optimizer.zero_grad()
                output = model(data)
                loss = mixup_criterion(criterion, output, target_a, target_b, lam)
            else:
                # 標準訓練
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = output.max(1)
            train_total += target.size(0)
            train_correct += predicted.eq(target).sum().item()
        
        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        # 驗證階段
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                
                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        # 記錄歷史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # 打印信息
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        # 學習率調整
        scheduler.step(val_loss)
        
        # Early Stopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    # 載入最佳模型
    if early_stopping.best_model is not None:
        model.load_state_dict(early_stopping.best_model)
    
    return model, history

print("訓練函數已定義，包含以下正則化技術：")
print("- Weight Decay (L2正則化)")
print("- Learning Rate Scheduling")
print("- Early Stopping")
print("- Mixup (可選)")
print("- 模型內建的Dropout和Batch Normalization")

## 練習題

### 1. Dropout實驗
在同一個網路上測試不同的dropout率(0, 0.3, 0.5, 0.7)，比較它們對訓練和驗證準確率的影響。

### 2. Batch Normalization位置
實驗Batch Normalization在激活函數前後的效果差異。

### 3. 數據增強組合
設計一個數據增強pipeline，結合至少3種不同的變換，並評估其效果。

### 4. Mixup vs Cutout
在相同的模型和數據集上比較Mixup和Cutout的效果。

### 5. 正則化組合
實驗不同正則化技術的組合：
- Dropout + Weight Decay
- Batch Normalization + Weight Decay
- Dropout + Batch Normalization + Weight Decay

### 6. 自定義數據增強
設計一個新的數據增強技術，並實現它。

## 總結

### 正則化技術選擇指南

| 問題 | 推薦技術 |
|------|----------|
| 過擬合嚴重 | Dropout, Weight Decay, Data Augmentation |
| 訓練不穩定 | Batch Normalization, Layer Normalization |
| 訓練緩慢 | Batch Normalization, 更好的初始化 |
| 數據量少 | 強數據增強, Mixup, Cutout |
| 模型太複雜 | 簡化架構, Dropout, Weight Decay |

### 實踐建議

1. **從簡單開始**：先嘗試基礎的正則化技術
2. **逐步添加**：一次添加一種技術，觀察效果
3. **監控指標**：同時關注訓練和驗證性能
4. **調整超參數**：每種技術都有超參數需要調整
5. **數據優先**：良好的數據增強往往比複雜的正則化更有效

### 常用組合

**標準組合**：
```python
Data Augmentation + Batch Normalization + Dropout + Weight Decay
```

**現代組合**：
```python
AutoAugment/RandAugment + Batch Normalization + Mixup/CutMix + Weight Decay
```

### 下一步學習

1. 更多數據增強技術（AutoAugment, RandAugment）
2. 模型可視化和解釋
3. 遷移學習和微調
4. 模型壓縮和優化
5. 實戰項目應用